# LSTM-PFD: Train All 35 Models on GPU

This notebook trains all fault diagnosis models using Colab's T4 GPU.

**Estimated total time: ~2.5 hours on T4**

| Batch | Models | Est. Time |
|---|---|---|
| CNN (5) | CNN1D, AttentionCNN, MultiScale | ~15 min |
| ResNet (10) | ResNet18/34/50, Wide, SE | ~40 min |
| Transformer (6) | SignalTransformer, ViT, PatchTST | ~30 min |
| EfficientNet (4) | B0-B3 | ~20 min |
| Hybrid (4) | CNN-LSTM, CNN-TCN | ~20 min |
| Advanced (6) | PINN, Spectrogram2D | ~25 min |

## Step 1: Clone Repository & Setup

In [ ]:
# Clone the repository (replace with your repo URL)
!git clone https://github.com/YOUR_USERNAME/LSTM_PFD.git
%cd LSTM_PFD
!git checkout fix/master-plan

In [ ]:
# Setup: verify GPU, install deps, check project structure
!bash scripts/colab/01_setup.sh

## Step 2: Generate Training Data (~1 min)

Generates 2,860 synthetic bearing vibration signals (11 fault types × 200 + 30% augmented).

In [ ]:
!python scripts/colab/02_generate_data.py --num-signals 200

## Step 3: Train Models (run each batch one at a time)

Each batch trains its models sequentially with:
- Mixed precision (FP16) for 2x speedup
- Early stopping (patience=15)
- Best-model checkpointing
- Results saved to `results/` as JSON

In [ ]:
# Batch 1: CNN Models (5 models, ~15 min)
# CNN1D, AttentionCNN1D, LightweightAttentionCNN, MultiScaleCNN1D, DilatedMultiScaleCNN
!python scripts/colab/03_train_batch1_cnn.py

In [ ]:
# Batch 2: ResNet Models (10 models, ~40 min)
# ResNet18/34/50, WideResNet 16-8/16-10/22-8/28-10, SE-ResNet 18/34/50
!python scripts/colab/04_train_batch2_resnet.py

In [ ]:
# Batch 3: Transformer Models (6 models, ~30 min)
# SignalTransformer, ViT-Tiny/Small/Base, PatchTST, TSMixer
!python scripts/colab/05_train_batch3_transformer.py

In [ ]:
# Batch 4: EfficientNet Models (4 models, ~20 min)
# EfficientNet-B0, B1, B2, B3
!python scripts/colab/06_train_batch4_efficientnet.py

In [ ]:
# Batch 5: Hybrid Models (4 models, ~20 min)
# CNN-LSTM, CNN-TCN, CNN-Transformer-Small, MultiScale-CNN
!python scripts/colab/07_train_batch5_hybrid.py

In [ ]:
# Batch 6: Advanced Models (6 models, ~25 min)
# HybridPINN, PhysicsConstrainedCNN, DualStreamCNN, ContrastiveClassifier,
# ResNet18-2D (spectrogram), EfficientNet2D-B0
!python scripts/colab/08_train_batch6_advanced.py

## Step 4: View Results Summary

In [ ]:
import json
from pathlib import Path

results_dir = Path('results')
all_results = []

for f in sorted(results_dir.glob('batch_*.json')):
    batch = json.loads(f.read_text())
    print(f"\n{'='*60}")
    print(f"Batch: {batch['batch_name']} ({batch['total_time_s']:.0f}s)")
    print(f"{'='*60}")
    for r in batch['results']:
        if 'error' in r:
            print(f"  FAIL  {r['model']}: {r['error'][:50]}")
        else:
            print(f"  {r['test_accuracy']:.4f}  {r['model']:<30} "
                  f"({r['params']:,} params, {r['epochs_trained']}ep, {r['training_time_s']:.0f}s)")
            all_results.append(r)

# Overall leaderboard
if all_results:
    print(f"\n\n{'='*60}")
    print("OVERALL LEADERBOARD (sorted by test accuracy)")
    print(f"{'='*60}")
    for i, r in enumerate(sorted(all_results, key=lambda x: x['test_accuracy'], reverse=True), 1):
        print(f"  {i:2d}. {r['test_accuracy']:.4f}  {r['model']:<30} ({r['params']:,} params)")

## Step 5: Push Results to GitHub

In [ ]:
# Configure git (replace with your info)
!git config user.email "your.email@example.com"
!git config user.name "Your Name"

# Add results (NOT checkpoints — too large for GitHub)
!git add results/ logs/
!git commit -m "feat(P3): training results for 35 models on Colab T4"
!git push

## Step 6: Save Checkpoints to Google Drive

Checkpoints are too large for GitHub. Save them to Google Drive instead.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
dest = Path('/content/drive/MyDrive/LSTM_PFD_checkpoints')
dest.mkdir(parents=True, exist_ok=True)

# Copy all checkpoints to Drive
src = Path('checkpoints')
for ckpt_dir in sorted(src.iterdir()):
    if ckpt_dir.is_dir():
        dest_dir = dest / ckpt_dir.name
        shutil.copytree(ckpt_dir, dest_dir, dirs_exist_ok=True)
        size = sum(f.stat().st_size for f in dest_dir.rglob('*')) / 1e6
        print(f"  [OK] {ckpt_dir.name} -> Drive ({size:.1f} MB)")

print(f"\nAll checkpoints saved to: {dest}")